# Plot the extended SCvH EOS

The original SCvH EOS table was extended to lower temperatures and pressures. However, these extended tables are in $(P, T)$ and cover a different range of pressure (and therefore have a different number of data points) for each isotherm.

### Import modules

In [ ]:
from __future__ import print_function
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import interpolate as interp
from scipy import optimize
import numpy as np

In [ ]:
%matplotlib widget
%config InlineBackend.figure_format = 'retina'

In [ ]:
mpl.rcParams['figure.dpi'] = 100

Convert units from cgs to SI:

In [ ]:
press_unit_si = 0.1        # erg/cm^3 in Pascal
spec_energy_unit_si = 1e4  # erg/g in J/kg

In [ ]:
input_file = "hydrogen_scvh_extended.data"

data = np.loadtxt(input_file) 

logT_table   = data[:, 0] 
logP_table   = data[:, 1]
frac_H2      = data[:, 2]
frac_H       = data[:, 3]
logrho_table = data[:, 4]
logu_table   = data[:, 5]
logs_table   = data[:, 6]

# All the SCvH EOS tables are tabulated along isotherms
logT_table_axis = np.unique(logT_table)
nT = np.size(logT_table_axis)

print("Number of isotherms: nT = {:}".format(nT))

# The number of grid points in P are different for each isotherm
logP_table_axis = list()
logrho_isotherm = list()
logu_isotherm = list()
logs_isotherm = list()

for logT in logT_table_axis:
    logP_table_axis.append(logP_table[np.where(logT_table == logT)])
    logrho_isotherm.append(logrho_table[np.where(logT_table == logT)])
    logu_isotherm.append(logu_table[np.where(logT_table == logT)])
    logs_isotherm.append(logs_table[np.where(logT_table == logT)])

logT_min = np.min(logT_table_axis)
logT_max = np.max(logT_table_axis)

print("logT_min = {:}".format(logT_min))
print("logT_max = {:}".format(logT_max))
print()

nP = list()
for logP in logP_table_axis:
    nP.append(np.size(logP))

table_rho_limit = list()

for i in range(nT):
    logrho_isotherm_min = np.min(logrho_isotherm[i])
    logrho_isotherm_max = np.max(logrho_isotherm[i])
    logT = logT_table_axis[i]
    
    table_rho_limit.append(np.array([logT, logrho_isotherm_min, logrho_isotherm_max]))
    #print("{:15.7E}{:15.7E}{:15.7E}".format(logT, logrho_isotherm_min, logrho_isotherm_max))

header = " nT = {:} (input file: {:})\n"\
         " {:>15s} {:>17s} {:>17s}".format(nT, input_file, "logT [K]", "logrhomin [g/cc]", "logrhomax [g/cc]")
    
np.savetxt("logrho_limit_pt.txt", table_rho_limit, header=header, fmt='%17.7e', delimiter=" ", comments="#")

In [ ]:
# Plot where the table is defined in (rho, T)
fig, ax = plt.subplots(1, 2)

x, y = fig.get_size_inches()

fig.set_size_inches(2*x, 1*y)

for i in range(nT):
    ax[0].plot(np.ones(nP[i])*logT_table_axis[i], logrho_isotherm[i], marker='.')
    
for i in range(nT):
    ax[1].plot(logrho_isotherm[i], np.ones(nP[i])*logT_table_axis[i], marker='.')